# 001 · Regular Expressions with re

> **Standard Library › Text Processing** · Python 3.14

The `re` module is Python's regular-expression engine — the counterpart of `System.Text.RegularExpressions`. The pattern syntax is almost identical to .NET's, but the API is shaped differently: module-level functions, `None` instead of an unsuccessful `Match`, and three different "match" functions.

**In this notebook you will learn:**

- why patterns are written as raw strings (`r"..."`)
- the difference between `match`, `search` and `fullmatch` (and how they map to `Regex.Match` with anchors)
- how to find all matches with `findall` / `finditer`, and work with groups and named groups
- how to replace with `sub` (string or function) and split with `split`
- flags (`IGNORECASE`, `MULTILINE`, `VERBOSE`), compiling, and greedy vs lazy quantifiers
- the classic gotcha: a failed match returns `None`

## .NET mental model

| .NET (`System.Text.RegularExpressions`) | Python (`re`) |
|---|---|
| `Regex.IsMatch(s, p)` | `re.search(p, s) is not None` |
| `Regex.Match(s, p)` → `Match` (check `.Success`) | `re.search(p, s)` → `Match` **or `None`** |
| `Regex.Match(s, "^...")` | `re.match(p, s)` (anchored at the start) |
| `Regex.Match(s, "^...$")` | `re.fullmatch(p, s)` |
| `Regex.Matches(s, p)` | `re.finditer(p, s)` / `re.findall(p, s)` |
| `Regex.Replace(s, p, r)` | `re.sub(p, r, s)` |
| `Regex.Split(s, p)` | `re.split(p, s)` |
| `new Regex(p, RegexOptions.Compiled)` | `re.compile(p)` |
| `RegexOptions.IgnoreCase` | `re.IGNORECASE` / `re.I` |
| `[GeneratedRegex(...)]` source generator | no equivalent |

Note the **argument order**: in Python the pattern comes first (`re.search(pattern, text)`), in .NET the input comes first (`Regex.Match(input, pattern)`).

## Raw strings for patterns

Regexes are full of backslashes. In a normal Python string, `\b` is a backspace character and `\d` only works by accident (unknown escapes are passed through, with a `SyntaxWarning`). Always write patterns as **raw strings** `r"..."` — the equivalent of a C# verbatim string `@"..."`.

```csharp
// C#
var rx = new Regex(@"\bcat\b");
```

In [ ]:
import re

print(len("\b"), len(r"\b"))          # 1 vs 2 — "\b" is a backspace char!
print(re.search("\bcat\b", "a cat sat"))   # None — searched for backspaces
print(re.search(r"\bcat\b", "a cat sat"))  # word boundary, as intended

## `match` vs `search` vs `fullmatch`

This trips up almost every newcomer. .NET has a single `Regex.Match` that searches anywhere; Python has three:

| Python | Looks for a match… | .NET equivalent |
|---|---|---|
| `re.search(p, s)` | anywhere in the string | `Regex.Match(s, p)` |
| `re.match(p, s)` | only at the **start** | `Regex.Match(s, "^(?:p)")` |
| `re.fullmatch(p, s)` | covering the **whole** string | `Regex.Match(s, @"^(?:p)\z")` |

In [ ]:
text = "order 42 shipped"

print(re.search(r"\d+", text))      # finds "42" in the middle
print(re.match(r"\d+", text))       # None — text doesn't START with digits
print(re.fullmatch(r"\d+", "42"))   # whole string is digits
print(re.fullmatch(r"\d+", "42a"))  # None

**Tip:** for validation ("is this whole string a valid product code?") use `fullmatch`. It avoids the classic `^...$` bug where `$` also matches before a trailing newline.

In [ ]:
code_pattern = r"[A-Z]{3}-\d{4}"
for candidate in ["ABC-1234", "ABC-12345", "xABC-1234", "ABC-1234\n"]:
    print(repr(candidate), bool(re.fullmatch(code_pattern, candidate)))

## The Match object — and the `None` gotcha

A successful call returns a `re.Match` object. `.group()` (or `m[0]`) is the matched text, `.start()`, `.end()` and `.span()` give positions.

```csharp
// C#
var m = Regex.Match("order 42", @"\d+");
if (m.Success) Console.WriteLine($"{m.Value} at {m.Index}");
```

In [ ]:
m = re.search(r"\d+", "order 42 shipped")
if m:                                  # Match objects are truthy; None is falsy
    print(m.group(), m[0])             # C#: m.Value
    print(m.start(), m.end(), m.span())  # C#: m.Index, m.Index + m.Length

⚠️ **Gotcha:** in .NET a failed `Regex.Match` returns a `Match` with `Success == false`, so `m.Value` is just `""`. In Python a failed match returns **`None`**, and calling `.group()` on it blows up with `AttributeError`. Always check first.

In [ ]:
m = re.search(r"\d+", "no digits here")
try:
    print(m.group())
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

# Idiomatic: the walrus operator tests and binds in one go
if (m := re.search(r"\d+", "order 7")) is not None:
    print("found", m[0])

## Finding all matches: `findall` and `finditer`

- `re.findall` returns a **list of strings** — simple, but you lose positions.
- `re.finditer` returns an **iterator of `Match` objects** — the true equivalent of `Regex.Matches`.

```csharp
// C#
foreach (Match m in Regex.Matches(text, @"\d+"))
    Console.WriteLine($"{m.Value} at {m.Index}");
```

In [ ]:
text = "3 apples, 12 pears and 7 plums"

print(re.findall(r"\d+", text))
for m in re.finditer(r"\d+", text):
    print(m[0], "at", m.start())

⚠️ **Gotcha:** `findall` changes shape when the pattern has groups. With one group it returns that group's text; with several it returns **tuples**. Use `finditer` if you want consistent `Match` objects.

In [ ]:
text = "alice=3, bob=5"
print(re.findall(r"\w+=\d+", text))        # no groups -> whole matches
print(re.findall(r"(\w+)=\d+", text))      # one group -> just the group
print(re.findall(r"(\w+)=(\d+)", text))    # two groups -> tuples

## Groups and named groups

Numbered groups work as in .NET. Named groups use `(?P<name>...)` — the **`P`** is required; the .NET-style `(?<name>...)` is a syntax error in `re`. Back-references to a named group are `(?P=name)`; in a replacement string they are `\g<name>`.

| .NET | Python |
|---|---|
| `(?<year>\d{4})` | `(?P<year>\d{4})` |
| `m.Groups["year"].Value` | `m["year"]` or `m.group("year")` |
| `\k<year>` (back-reference) | `(?P=year)` |
| `${year}` (in replacement) | `\g<year>` |

In [ ]:
m = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", "due 2026-03-15")
print(m["year"], m.group("month"), m[3])   # by name or by number
print(m.groups())                          # all groups as a tuple
print(m.groupdict())                       # C#: no direct equivalent — handy!

try:
    re.compile(r"(?<year>\d{4})")          # .NET syntax
except re.error as e:
    print(f"{type(e).__name__}: {e}")

Non-capturing groups `(?:...)` are the same as in .NET:

In [ ]:
m = re.fullmatch(r"(?:\d{3}-)?(\d{4})", "555-1234")
print(m.groups())   # only one capturing group

## Replacing with `sub`

`re.sub(pattern, repl, string)` replaces every match (pass `count=` to limit). The replacement can be:

- a **string**, using `\1` or `\g<name>` for groups (.NET: `$1`, `${name}`)
- a **function** that receives a `Match` and returns the replacement (.NET: `MatchEvaluator`)

```csharp
// C#
var iso = Regex.Replace(s, @"(\d{2})/(\d{2})/(\d{4})", "$3-$2-$1");
var up  = Regex.Replace(s, @"\b\w", m => m.Value.ToUpper());
```

In [ ]:
s = "start 15/03/2026, end 01/04/2026"
print(re.sub(r"(\d{2})/(\d{2})/(\d{4})", r"\3-\2-\1", s))
print(re.sub(r"(?P<d>\d{2})/(?P<m>\d{2})/(?P<y>\d{4})", r"\g<y>-\g<m>-\g<d>", s))

In [ ]:
def double(m: re.Match) -> str:
    return str(int(m[0]) * 2)

print(re.sub(r"\d+", double, "3 apples and 12 pears"))
print(re.sub(r"\b\w", lambda m: m[0].upper(), "hello regex world"))
print(re.subn(r"\d", "#", "pin 1234"))   # subn also returns the number of replacements

## Splitting

`re.split` is like `Regex.Split`. As in .NET, capturing groups in the pattern are **kept** in the result.

In [ ]:
print(re.split(r"[,;]\s*", "red, green;blue,  yellow"))
print(re.split(r"([,;])", "a,b;c"))          # separators kept, like .NET
print(re.split(r"\s+", "  leading space"))   # note the empty first item

**Tip:** for splitting on plain whitespace you don't need a regex: `"  a  b ".split()` (no arguments) splits on runs of whitespace and drops empty items.

## Flags

| .NET `RegexOptions` | Python flag | Inline |
|---|---|---|
| `IgnoreCase` | `re.IGNORECASE` / `re.I` | `(?i)` |
| `Multiline` | `re.MULTILINE` / `re.M` | `(?m)` |
| `Singleline` | `re.DOTALL` / `re.S` | `(?s)` |
| `IgnorePatternWhitespace` | `re.VERBOSE` / `re.X` | `(?x)` |
| `CultureInvariant` | n/a — use `re.ASCII` / `re.A` to make `\w`, `\d` ASCII-only | `(?a)` |

Combine flags with `|`, just like .NET flag enums.

In [ ]:
log = "INFO start\nerror disk full\nInfo done"

print(re.findall(r"^info.*", log, flags=re.IGNORECASE))               # only line 1: ^ = start of string
print(re.findall(r"^info.*", log, flags=re.IGNORECASE | re.MULTILINE)) # ^ = start of each line

`re.VERBOSE` lets you spread a pattern over lines with comments — great for anything non-trivial. Whitespace in the pattern is ignored, so escape literal spaces (`\ `) or use `\s`.

In [ ]:
phone = re.compile(r"""
    ^\(?(?P<area>\d{3})\)?   # area code, optional parentheses
    [\s.-]?                  # optional separator
    (?P<exchange>\d{3})
    [\s.-]?
    (?P<line>\d{4})$
""", re.VERBOSE)

for p in ["(555) 123-4567", "555.123.4567", "5551234567", "55-1234"]:
    m = phone.match(p)
    print(p, "->", m.groupdict() if m else None)

## Compiling and caching

`re.compile(pattern, flags)` returns a `re.Pattern` object with the same methods (`search`, `match`, `fullmatch`, `findall`, `finditer`, `sub`, `split`) — the analogue of creating a `Regex` instance. Pattern methods also accept `pos`/`endpos` arguments.

```csharp
// C#
private static readonly Regex Word = new(@"\w+", RegexOptions.Compiled);
```

In [ ]:
WORD = re.compile(r"\w+")
print(WORD.findall("the quick brown fox"))
print(WORD.pattern, WORD.flags & re.UNICODE != 0)   # str patterns are Unicode by default

Do you *need* to compile? Not for performance in most code: the module-level functions (`re.search(...)` etc.) keep an internal cache of recently compiled patterns, much like .NET's static `Regex` methods cache (`Regex.CacheSize`). Compile anyway when it improves readability — a named module-level constant documents intent and keeps flags next to the pattern.

There is **no equivalent of .NET's `[GeneratedRegex]` source generator** or `RegexOptions.Compiled` IL emission: Python always uses its own backtracking engine, compiled to internal bytecode.

## Greedy vs lazy quantifiers

Identical to .NET: `*`, `+`, `?`, `{n,m}` are greedy; append `?` for lazy (`*?`, `+?`, `??`).

In [ ]:
html = "<b>bold</b> and <i>italic</i>"

print(re.findall(r"<.+>", html))    # greedy: one giant match
print(re.findall(r"<.+?>", html))   # lazy: each tag
print(re.findall(r"<[^>]+>", html)) # often better: a negated class

## Escaping user input

When building a pattern from data, escape it with `re.escape` (.NET: `Regex.Escape`):

In [ ]:
term = "price (USD)?"
print(re.escape(term))
print(bool(re.search(re.escape(term), "what is the price (USD)?")))

⚠️ **Gotcha:** some .NET regex features are not available in `re`: no variable-length lookbehind (Python requires fixed-width lookbehind), no balancing groups, and no `RegexOptions.RightToLeft`. Atomic groups `(?>...)` and possessive quantifiers `*+` *are* supported (since 3.11). (Errors are `re.error`, also available as `re.PatternError` since 3.13.) A timeout like `Regex.MatchTimeout` does not exist either — beware catastrophic backtracking on untrusted input.

In [ ]:
try:
    re.compile(r"(?<=\d+)px")      # variable-width lookbehind
except re.error as e:
    print(f"{type(e).__name__}: {e}")

print(re.findall(r"(?<=\d)px", "10px 20em 3px"))   # fixed-width is fine

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Pattern literal | `@"\d+"` | `r"\d+"` |
| Search anywhere | `Regex.Match(s, p)` | `re.search(p, s)` |
| Anchored at start | `Regex.Match(s, "^p")` | `re.match(p, s)` |
| Whole-string validation | `Regex.IsMatch(s, "^p$")` | `re.fullmatch(p, s)` |
| Failed match | `m.Success == false` | returns `None` |
| All matches | `Regex.Matches(s, p)` | `re.finditer(p, s)` / `re.findall(p, s)` |
| Named group | `(?<name>...)`, `m.Groups["name"]` | `(?P<name>...)`, `m["name"]` |
| Replace | `Regex.Replace(s, p, "$1")` | `re.sub(p, r"\1", s)` |
| Replace with function | `MatchEvaluator` | `re.sub(p, func, s)` |
| Split | `Regex.Split(s, p)` | `re.split(p, s)` |
| Options | `RegexOptions.IgnoreCase \| Multiline` | `re.I \| re.M` |
| Reusable regex | `new Regex(p)` | `re.compile(p)` |
| Escape | `Regex.Escape(s)` | `re.escape(s)` |

## Exercises

1. Write a function `is_valid_sku(s)` that returns `True` only if the **whole** string is two uppercase letters, a dash and 3–5 digits (e.g. `"AB-123"`).
2. Extract all `key=value` pairs from `"host=localhost port=8080 debug=true"` into a `dict` using named groups.
3. Convert `snake_case` identifiers in `"user_id and order_total_amount"` to `camelCase` using `re.sub` with a function.
4. Using a verbose pattern, parse the log line `"2026-03-15 10:42:07 WARN disk almost full"` into date, time, level and message.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import re

# 1 — fullmatch anchors both ends
def is_valid_sku(s: str) -> bool:
    return re.fullmatch(r"[A-Z]{2}-\d{3,5}", s) is not None

print(is_valid_sku("AB-123"), is_valid_sku("AB-12"), is_valid_sku("xAB-123"))

# 2 — finditer + named groups
text = "host=localhost port=8080 debug=true"
pairs = {m["key"]: m["value"] for m in re.finditer(r"(?P<key>\w+)=(?P<value>\S+)", text)}
print(pairs)

# 3 — function replacement: "_x" -> "X"
print(re.sub(r"_([a-z])", lambda m: m[1].upper(), "user_id and order_total_amount"))

# 4 — VERBOSE pattern
LOG = re.compile(r"""
    (?P<date>\d{4}-\d{2}-\d{2})\s+
    (?P<time>\d{2}:\d{2}:\d{2})\s+
    (?P<level>[A-Z]+)\s+
    (?P<message>.*)
""", re.VERBOSE)
print(LOG.fullmatch("2026-03-15 10:42:07 WARN disk almost full").groupdict())
```

</details>